In [0]:
catalogo = "medalhao"
bronze_db_name = "bronze"
silver_db_name = "silver"
gold_db_name = "gold"

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import Row

def ingest_csv(nome_arquivo, nome_tabela):
    try:
        table_name = nome_tabela 
        landing_path = f"/Volumes/medalhao/landing/raw_data/{nome_arquivo}" #caminho da base de dados
        df = spark.read.csv(landing_path, header=True, inferSchema=True) #lendo a base de dados, inferSchema=True faz com que o spark entenda o tipo de dados e header que tem cabeçalho
        if df.count() == 0:
            raise ValueError (f"O arquivo {nome_arquivo} está Vazio.") #verificando se o arquivo tem algum conteúdo
        df_metadata = df.withColumn("timestamp_ingestion", F.current_timestamp()) #adicionando uma coluna com a data de ingestão dos dados
        df_metadata.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{table_name}") #escrevendo a base de dados no formato delta, overwrite faz com que ele substitua o conteúdo da tabela caso ela já exista e saveAsTable faz com que ele salve a base de dados no formato delta no caminho especificado
    except Exception as e:
        print(f"Ocorreu um erro ao processar o arquivo {nome_arquivo}: {str(e)}")

#ingestão dos .CSVs
ingest_csv("olist_customers_dataset.csv", "bronze.tb_customers")
ingest_csv("olist_geolocation_dataset.csv", "bronze.tb_geolocalizacao")
ingest_csv("olist_order_items_dataset.csv", "bronze.tb_order_items")
ingest_csv("olist_order_payments_dataset.csv", "bronze.tb_order_payments")
ingest_csv("olist_order_reviews_dataset.csv", "bronze.tb_order_reviews")
ingest_csv("olist_orders_dataset.csv", "bronze.tb_orders")
ingest_csv("olist_products_dataset.csv", "bronze.tb_products")
ingest_csv("olist_sellers_dataset.csv", "bronze.tb_sellers")
ingest_csv("product_category_name_translation.csv", "bronze.tb_product_category_name_translation")

print("Ingestão concluída com sucesso")


Ingestao de API

In [0]:
import requests

dbutils.widgets.text("data_inicio", "09-04-2016") #cria uma widget e coloca um valor padrão com base no dataset
dbutils.widgets.text("data_fim", "10-17-2018")

data_inicio = dbutils.widgets.get("data_inicio") #pega o valor da widget, o widget é criada no notebook de ingestão
data_fim = dbutils.widgets.get("data_fim")

data_inicio_formatada = data_inicio
data_fim_formatada = data_fim

url = (f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json")

resposta = requests.get(url, timeout=30) #requisição para url e evita que o código fique travado

resposta.raise_for_status() #verifica se a requisição foi bem sucedida

dados = resposta.json() #transforma a resposta em json

registros = dados.get("value", []) #pega os valores da chave value pois a API trabalha com essa chave

schema = StructType([
    StructField("dataHoraCotacao", StringType(), True), #estrutura(nome da coluna, tipo da coluna, True aceita nulo)
    StructField("cotacaoCompra", FloatType(), True)
]) #schema da tabela

#uma lita q inicia todos os objetos Row
rows = [Row(dataHoraCotacao = r["dataHoraCotacao"], 
            cotacaoCompra = float(r["cotacaoCompra"])
            )
        for r in registros
] 
df_cotacao = spark.createDataFrame(rows, schema=schema) #cria um dataframe com os dados da lista rows
df_cotacao = df_cotacao.withColumn("timestamp_ingestion", F.current_timestamp()) #adicionando uma coluna com a data de ingestão dos dados
df_cotacao = df_cotacao.withColumn("dataHoraCotacao", F.to_timestamp("dataHoraCotacao")) #transformando a coluna dataHoraCotacao em timestamp
(df_cotacao.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
.saveAsTable(f"{catalogo}.{bronze_db_name}.tb_cotacao_dolar")
 )
 